# InMemory Backend — Zero-Dependency Caching with Medha

Medha 0.2.0 ships with `InMemoryBackend`: a **pure-Python in-process vector store**
that requires no external services. It is the fastest way to get started and the
recommended backend for unit tests and CI pipelines.

| Property | InMemoryBackend | QdrantBackend |
|---|---|---|
| External deps | none | `qdrant-client` |
| Infrastructure | none | Qdrant process/cloud |
| Persistence | no (in-process only) | yes |
| Search | linear cosine scan O(n) | HNSW ANN O(log n) |
| Best for | tests, CI, demos, serverless | production |

This notebook covers:

1. **Getting started** — `backend_type="memory"` in Settings
2. **Direct instantiation** — passing `InMemoryBackend()` explicitly
3. **Shared backend, multiple collections** — one store, many Medha instances
4. **Using it for tests** — deterministic, no infra, no cleanup required
5. **Limitations** — what to be aware of in production

**Requirements:** `pip install "medha-archai[fastembed]"`

## Cell 1: Imports

In [ ]:
import time

from medha import Medha, InMemoryBackend, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import QueryTemplate

## Cell 2: Getting Started — `backend_type="memory"`

The simplest way to use `InMemoryBackend` is via `Settings(backend_type="memory")`.
Medha automatically creates and wires the backend — no extra imports needed.

Note that you do **not** need `qdrant_mode` when using this backend.

In [ ]:
settings = Settings(backend_type="memory")   # no qdrant_mode required
embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

medha = Medha("inmem_demo", embedder=embedder, settings=settings)
await medha.start()

pairs = [
    ("How many users are there?",            "SELECT COUNT(*) FROM users"),
    ("List all active users",                 "SELECT * FROM users WHERE status = 'active'"),
    ("What is the average salary?",           "SELECT AVG(salary) FROM employees"),
    ("Show top 10 products by price",         "SELECT * FROM products ORDER BY price DESC LIMIT 10"),
    ("Count orders placed today",             "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
]

for q, sql in pairs:
    await medha.store(q, sql)

print(f"Stored {len(pairs)} entries in InMemoryBackend.")
print(f"Backend type: {type(medha._backend).__name__}")

await medha.close()

## Cell 3: Full Waterfall — All Tiers Work the Same

`InMemoryBackend` exposes exactly the same `VectorStorageBackend` interface as
`QdrantBackend`. All four cache tiers (L1, template, exact vector, semantic)
work identically — only the underlying similarity computation differs.

In [ ]:
settings = Settings(backend_type="memory")
embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

medha = Medha("inmem_demo", embedder=embedder, settings=settings)
await medha.start()

pairs = [
    ("How many users are there?",            "SELECT COUNT(*) FROM users"),
    ("List all active users",                 "SELECT * FROM users WHERE status = 'active'"),
    ("What is the average salary?",           "SELECT AVG(salary) FROM employees"),
    ("Show top 10 products by price",         "SELECT * FROM products ORDER BY price DESC LIMIT 10"),
    ("Count orders placed today",             "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
]

for q, sql in pairs:
    await medha.store(q, sql)

test_queries = [
    ("How many users are there?",        "exact repeat"),
    ("How many users are there?",        "L1 on second call"),
    ("What's the total user count?",     "semantic paraphrase"),
    ("Show the priciest products",       "semantic paraphrase"),
    ("Something completely unrelated",   "should miss"),
]

print(f"  {'Question':<45s}  {'Strategy':<20s}  {'Conf':>6s}  Note")
print("  " + "-" * 90)

for question, note in test_queries:
    t0 = time.perf_counter()
    hit = await medha.search(question)
    elapsed = (time.perf_counter() - t0) * 1000
    conf = f"{hit.confidence:.3f}" if hit.confidence else "  n/a"
    print(f"  {question:<45s}  {hit.strategy.value:<20s}  {conf}  ({note})")

await medha.close()

## Cell 4: Direct Instantiation — Passing `InMemoryBackend()` Explicitly

You can also instantiate `InMemoryBackend` directly and pass it to `Medha`.
This is useful when you want to inspect the backend state or share it across
multiple `Medha` instances (see Cell 5).

In [ ]:
backend = InMemoryBackend()   # create backend explicitly
embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")
settings = Settings()         # settings still needed for thresholds etc.

medha = Medha(
    collection_name="explicit_backend",
    embedder=embedder,
    backend=backend,           # pass backend directly — backend_type is ignored
    settings=settings,
)
await medha.start()

await medha.store("Total registered users", "SELECT COUNT(*) FROM users")
await medha.store("Monthly revenue", "SELECT SUM(amount) FROM invoices WHERE MONTH(created_at)=MONTH(NOW())")

# Inspect backend directly
count = await backend.count("explicit_backend")
print(f"InMemoryBackend collection 'explicit_backend': {count} entries")

hit = await medha.search("How many users registered?")
print(f"Search: strategy={hit.strategy.value}, query={hit.generated_query!r}")

await medha.close()

## Cell 5: Shared Backend — Multiple Collections, One Store

A single `InMemoryBackend` instance can serve multiple `Medha` instances.
Each uses a different collection name — full data isolation, single memory footprint.

This mirrors the multi-tenant pattern from `09_multi_tenant.ipynb`, but without
any external infrastructure.

In [ ]:
shared_backend = InMemoryBackend()
shared_embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")
base_settings = Settings()  # shared thresholds

# Two applications sharing one InMemoryBackend
medha_sql = Medha("app_sql", embedder=shared_embedder, backend=shared_backend, settings=base_settings)
medha_cypher = Medha("app_cypher", embedder=shared_embedder, backend=shared_backend, settings=base_settings)

await medha_sql.start()
await medha_cypher.start()

await medha_sql.store("How many users?", "SELECT COUNT(*) FROM users")
await medha_cypher.store("How many users?", "MATCH (u:User) RETURN COUNT(u)")

# Isolation: same question, different backends
question = "User count"
hit_sql = await medha_sql.search(question)
hit_cypher = await medha_cypher.search(question)

print(f"SQL app:    {hit_sql.generated_query}")
print(f"Cypher app: {hit_cypher.generated_query}")
print(f"\nShared backend collections: {list(shared_backend._store.keys())}")
print(f"SQL entries:    {await shared_backend.count('app_sql')}")
print(f"Cypher entries: {await shared_backend.count('app_cypher')}")

await medha_sql.close()
await medha_cypher.close()

## Cell 6: Template Matching Works Too

Templates are stored in a separate collection (`__medha_templates_<name>`) internally.
`InMemoryBackend` handles multiple collections transparently, so template matching
works exactly as in the Qdrant demos.

In [ ]:
settings = Settings(backend_type="memory", score_threshold_template=0.35)
medha = Medha("template_demo", embedder=FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache"), settings=settings)
await medha.start()

templates = [
    QueryTemplate(
        intent="employees_by_dept",
        template_text="List employees in {department}",
        query_template="SELECT * FROM employees WHERE department = '{department}'",
        parameters=["department"],
        parameter_patterns={"department": r"(Engineering|Marketing|Sales|HR)"},
    ),
]

await medha.load_templates(templates)

test_questions = [
    "Show employees in Engineering",
    "List all staff in Marketing",
    "Who works in Sales?",
]

for q in test_questions:
    hit = await medha.search(q)
    print(f"  [{hit.strategy.value:16s}] {q!r}")
    if hit.generated_query:
        print(f"    -> {hit.generated_query}")

await medha.close()


## Cell 7: Use Case — Unit Tests Without Mocks

The main motivation for `InMemoryBackend` is **test isolation**: no mocking,
no Qdrant process, no network — just deterministic in-process state that is
discarded when the test ends.

This cell simulates a pytest-style test using `InMemoryBackend`.

In [ ]:
# Simulates what a pytest fixture + test would look like

async def make_medha(collection="test_col"):
    """Factory used in tests: creates a fresh InMemoryBackend each time."""
    settings = Settings(backend_type="memory")
    medha = Medha(collection, embedder=FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache"), settings=settings)
    await medha.start()
    return medha


async def test_store_and_retrieve():
    medha = await make_medha()
    await medha.store("Count all users", "SELECT COUNT(*) FROM users")
    hit = await medha.search("Count all users")
    assert hit.strategy in (SearchStrategy.EXACT_MATCH, SearchStrategy.L1_CACHE)
    assert hit.generated_query == "SELECT COUNT(*) FROM users"
    await medha.close()
    return "PASS"


async def test_miss_returns_no_match():
    medha = await make_medha()
    hit = await medha.search("something completely unrelated")
    assert hit.strategy == SearchStrategy.NO_MATCH
    await medha.close()
    return "PASS"


async def test_max_question_length_guard():
    settings = Settings(backend_type="memory", max_question_length=64)
    medha = Medha("guard_test", embedder=FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache"), settings=settings)
    await medha.start()
    hit = await medha.search("a" * 100)
    assert hit.strategy == SearchStrategy.ERROR
    await medha.close()
    return "PASS"


results = {
    "test_store_and_retrieve":     await test_store_and_retrieve(),
    "test_miss_returns_no_match":  await test_miss_returns_no_match(),
    "test_max_question_length_guard": await test_max_question_length_guard(),
}

for name, result in results.items():
    print(f"  {result}  {name}")

## Cell 8: Performance Characteristics

`InMemoryBackend` uses **linear cosine scan** (O(n)) — every search compares
the query vector against all stored vectors. For small collections (< 10k entries)
this is fast; for large collections, use `QdrantBackend` (HNSW, O(log n)).

In [ ]:
import asyncio

# Measure search time as collection grows
sizes = [10, 100, 500, 1000]

print(f"  {'Size':>6s}  {'Search time':>12s}  {'per-entry':>12s}")
print("  " + "-" * 38)

for n in sizes:
    settings = Settings(backend_type="memory")
    medha = Medha(f"perf_{n}", embedder=FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache"), settings=settings)
    await medha.start()

    # Batch-store n synthetic entries
    batch = [
        {"question": f"Synthetic question number {i} about data",
         "generated_query": f"SELECT * FROM table_{i}"}
        for i in range(n)
    ]
    await medha.store_batch(batch)

    # Warm L1 bypass
    await medha.clear_caches()

    # Time a search (3 trials, take median)
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        await medha.search("How many records in the table?")
        times.append((time.perf_counter() - t0) * 1000)
    median_ms = sorted(times)[1]

    print(f"  {n:>6d}  {median_ms:>10.2f}ms  {median_ms/n*1000:>9.2f}µs")
    await medha.close()

print("\nFor n > 10k, switch to backend_type='qdrant' for HNSW O(log n) search.")

## Cell 9: Limitations Summary

| Limitation | Detail | Mitigation |
|---|---|---|
| No persistence | Data is lost when the process exits | Use `QdrantBackend` or `PgVectorBackend` in production |
| Linear scan O(n) | Slow for n > 10k | Use `QdrantBackend` (HNSW) for large collections |
| No quantization | Full float32 vectors in RAM | `QdrantBackend` supports scalar/binary quantization |
| Single-process | Cannot be shared across pods | Use `QdrantBackend` or `PgVectorBackend` for horizontal scale |
| No payload indexes | `query_hash` lookup is O(n) | Acceptable for small collections |

**Recommendation:** Use `InMemoryBackend` for development, testing, and prototyping.
Switch to `QdrantBackend` or `PgVectorBackend` when you need persistence or scale.

In [ ]:
print("InMemoryBackend summary:")
print("  [+] Zero external dependencies")
print("  [+] Instant startup (no connection, no pool)")
print("  [+] Full VectorStorageBackend interface (same as Qdrant/pgvector)")
print("  [+] Safe for CI/CD pipelines without infrastructure")
print("  [-] No persistence (data lost on process exit)")
print("  [-] Linear O(n) search — not suitable for n > 10k in production")
print("  [-] Not shareable across processes")